In [ ]:
import json, os, glob
from collections import defaultdict

CASE_DIR = os.path.abspath(os.path.join("..", "cases", "murder"))
FIGURES_DIR = os.path.abspath(os.path.join("figures", "interleaved"))
os.makedirs(FIGURES_DIR, exist_ok=True)

MODE_DIRS = {"SBS": "outputs_interleaved", "EOS": "outputs_eos_interleaved"}

MODEL_NAMES = {
    "claude-3-haiku-20240307": "Claude 3 Haiku",
    "claude-3-5-haiku-20241022": "Claude 3.5 Haiku",
    "claude-3-7-sonnet-20250219": "Claude 3.7 Sonnet",
    "claude-sonnet-4-20250514": "Claude 4 Sonnet",
    "claude-sonnet-4-6": "Claude Sonnet 4.6",
    "gemini-2.0-flash": "Gemini 2.0 Flash",
    "gemini-2.5-flash": "Gemini 2.5 Flash",
    "gemini-3-flash-preview": "Gemini 3 Flash",
    "gpt-3.5-turbo": "GPT 3.5 Turbo",
    "gpt-4o": "GPT 4o",
    "gpt-5": "GPT 5",
    "gpt-5.4": "GPT 5.4",
    "meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8": "LLaMA 4 Maverick",
    "Qwen/Qwen2.5-72B-Instruct-Turbo": "Qwen 2.5 72B",
}

def scrape_counts(case_dir, out_dir):
    counts = defaultdict(lambda: {"dp": [0, 0], "pd": [0, 0]})
    for order in ["dp", "pd"]:
        order_path = os.path.join(case_dir, out_dir, order)
        if not os.path.isdir(order_path):
            continue
        for path in glob.glob(os.path.join(order_path, "**", "judgments_run*.json"), recursive=True):
            rel = os.path.relpath(os.path.dirname(path), order_path)
            with open(path) as f:
                d = json.load(f)
            if not isinstance(d[-1], bool):
                continue
            if d[-1] is True:
                counts[rel][order][0] += 1
            else:
                counts[rel][order][1] += 1
    return counts

data = {}
for mode, out_dir in MODE_DIRS.items():
    for model_key, orders in scrape_counts(CASE_DIR, out_dir).items():
        display = MODEL_NAMES.get(model_key, model_key)
        if display not in data:
            data[display] = {}
        data[display][mode] = {"DP": orders["dp"], "PD": orders["pd"]}

print(f"Loaded {len(data)} models")
for m, v in sorted(data.items()):
    print(f"  {m}: {v}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

guilty_color = "#2c2f7b"
not_guilty_color = "#a3a5d9"

def proportions(vals):
    total = sum(vals)
    return np.array(vals) / total if total > 0 else np.array([0.0, 0.0])

for model, conditions in sorted(data.items()):
    fig, axes = plt.subplots(1, 2, figsize=(8, 4), sharey=True)
    plt.subplots_adjust(wspace=0.25)

    for j, mode in enumerate(["EOS", "SBS"]):
        ax = axes[j]
        if mode not in conditions:
            ax.set_title(mode + "\n(no data)", fontsize=12, fontweight="bold")
            continue

        dp = proportions(conditions[mode]["DP"])
        pd = proportions(conditions[mode]["PD"])
        x = np.arange(2)
        width = 0.6

        ax.bar(x, [dp[0], pd[0]], width, color=guilty_color, label="Guilty" if j == 0 else "")
        ax.bar(x, [dp[1], pd[1]], width, bottom=[dp[0], pd[0]], color=not_guilty_color, label="Not Guilty" if j == 0 else "")

        ax.set_title(mode, fontsize=12, fontweight="bold")
        ax.set_xticks(x)
        ax.set_xticklabels(["DP", "PD"], fontsize=10)
        ax.set_ylim(0, 1)
        ax.set_yticks([0, 0.25, 0.5, 0.75, 1])
        ax.set_yticklabels(["0%", "25%", "50%", "75%", "100%"], fontsize=9)
        ax.grid(axis="y", linestyle="--", alpha=0.4)
        if j == 0:
            ax.set_ylabel("Proportion of Verdict", fontsize=10)

    axes[0].legend(title="Verdict", loc="upper right")
    fig.suptitle(f"{model} — Verdict Proportions (Interleaved)", fontsize=13, fontweight="bold", y=1.02)

    filename = model.replace(" ", "_").replace(".", "") + "_interleaved.png"
    filepath = os.path.join(FIGURES_DIR, filename)
    plt.tight_layout()
    plt.savefig(filepath, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved {filepath}")